In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

class TinyPolicyModel(nn.Module):
    """Stand-in for a causal LM. Real usage: AutoModelForCausalLM + TRL's DPOTrainer."""
    def __init__(self, vocab_size=100, d_model=32, seq_len=10):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.GRU(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        h = self.embed(input_ids)
        out, _ = self.rnn(h)
        return self.head(out)   # logits: (batch, seq_len, vocab_size)

def sequence_log_prob(model, input_ids, response_ids):
    """Sums log p(token) over the response tokens under `model`."""
    logits = model(input_ids)
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = torch.gather(log_probs, 2, response_ids.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum(dim=-1)

def dpo_loss(policy_model, ref_model, prompt_ids, chosen_ids, rejected_ids, beta=0.1):
    """Direct Preference Optimization loss (Rafailov et al.) using a frozen
    reference model and the trainable policy model -- no reward model, no RL."""
    policy_chosen_logp = sequence_log_prob(policy_model, prompt_ids, chosen_ids)
    policy_rejected_logp = sequence_log_prob(policy_model, prompt_ids, rejected_ids)

    with torch.no_grad():
        ref_chosen_logp = sequence_log_prob(ref_model, prompt_ids, chosen_ids)
        ref_rejected_logp = sequence_log_prob(ref_model, prompt_ids, rejected_ids)

    policy_logratios = policy_chosen_logp - policy_rejected_logp
    ref_logratios = ref_chosen_logp - ref_rejected_logp

    logits = beta * (policy_logratios - ref_logratios)
    loss = -F.logsigmoid(logits).mean()
    return loss

def train_dpo(steps=5, vocab_size=100, seq_len=10, batch_size=8):
    policy_model = TinyPolicyModel(vocab_size, seq_len=seq_len)
    ref_model = copy.deepcopy(policy_model)          # frozen reference model
    for p in ref_model.parameters():
        p.requires_grad = False

    optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-3)

    for step in range(steps):
        prompt_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
        chosen_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
        rejected_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

        loss = dpo_loss(policy_model, ref_model, prompt_ids, chosen_ids, rejected_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"step {step}: DPO loss={loss.item():.4f}")

    return policy_model

if __name__ == "__main__":
    torch.manual_seed(0)
    train_dpo()

---
## Task 13: Direct Preference Optimization (DPO) and Pairwise Reward Alignment